In [13]:
# =========================================================
# STEP 1: RESOLVE ABSOLUTE PATHS DYNAMICALLY
# =========================================================
import sys
import os
import pandas as pd
import numpy as np

# Find where this notebook file actually lives, then locate the project root folder
notebook_dir = os.getcwd()
if os.path.basename(notebook_dir) == 'notebooks':
    PROJECT_ROOT = os.path.abspath(os.path.join(notebook_dir, '..'))
else:
    # If already at root, use current directory
    PROJECT_ROOT = os.path.abspath(notebook_dir)

# Ensure Python can see your 'src' module folder 
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Construct precise, absolute locations for all your files
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# Make sure directory structures are physically built
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"🎯 Project Root mapped to absolute path: {PROJECT_ROOT}")
print(f"📁 Reading files explicitly from: {RAW_DIR}")

# =========================================================
# STEP 2: GENERATE POLICY SHOCK DATA IF MISSING
# =========================================================
policy_file_path = os.path.join(PROCESSED_DIR, "policy_events.csv")
if not os.path.exists(policy_file_path):
    print("📢 policy_events.csv not found. Creating it dynamically...")
    csv_content = """date,event,dummy
2021-01-01,Phase IV begins,1
2022-02-24,Russia-Ukraine war,1
2022-05-18,REPowerEU announcement,1
2023-01-01,MSR strengthening effective,1
2023-04-18,ETS reform (Fit for 55 final),1
2024-01-01,ETS2 framework confirmed,1"""
    with open(policy_file_path, "w") as f:
        f.write(csv_content)
    print("✅ Successfully generated policy_events.csv!")

# =========================================================
# STEP 3: FEATURE ENGINEERING MATRIX PIPELINE
# =========================================================
from src.features import build_feature_matrix

print("Loading raw files via absolute machine mapping...")
eua = pd.read_csv(os.path.join(RAW_DIR, "raw_eua.csv"), index_col=0, parse_dates=True)
ttf = pd.read_csv(os.path.join(RAW_DIR, "raw_ttf.csv"), index_col=0, parse_dates=True)
coal = pd.read_csv(os.path.join(RAW_DIR, "raw_coal.csv"), index_col=0, parse_dates=True)
weather = pd.read_csv(os.path.join(RAW_DIR, "raw_weather.csv"), index_col=0, parse_dates=True)
gen = pd.read_csv(os.path.join(RAW_DIR, "raw_generation.csv"), index_col=0, parse_dates=True)

print("Executing build_feature_matrix pipeline...")
df = build_feature_matrix(eua, ttf, coal, gen, weather, None)

# Save matrix to processed using absolute pathing
final_matrix_path = os.path.join(PROCESSED_DIR, "final_feature_matrix.csv")
df.to_csv(final_matrix_path)
print(f"✅ Feature matrix successfully created with shape: {df.shape}")
print(f"💾 Written directly to: {final_matrix_path}")

# =========================================================
# STEP 4: AUGMENTED DICKEY-FULLER STATIONARITY TESTS
# =========================================================
from statsmodels.tsa.stattools import adfuller

def adf_report(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"--- ADF Test: {name} ---")
    print(f"  p-value: {result[1]:.4f}")
    if result[1] < 0.05:
        print("  → STATUS: STATIONARY (Safe for baseline regressions!)\n")
    else:
        print("  → STATUS: NON-STATIONARY (Warning: Still contains stochastic trends)\n")

print("\nRunning Econometric Diagnostics...")
adf_report(df["eua_price"], "EUA Price Levels (Raw)")
adf_report(df["d_log_eua"], "Δ Log EUA Returns (Stationary-Transformed)")

🎯 Project Root mapped to absolute path: /Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting
📁 Reading files explicitly from: /Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting/data/raw
Loading raw files via absolute machine mapping...
Executing build_feature_matrix pipeline...
✅ Feature matrix successfully created with shape: (1283, 22)
💾 Written directly to: /Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting/data/processed/final_feature_matrix.csv

Running Econometric Diagnostics...
--- ADF Test: EUA Price Levels (Raw) ---
  p-value: 0.0000
  → STATUS: STATIONARY (Safe for baseline regressions!)

--- ADF Test: Δ Log EUA Returns (Stationary-Transformed) ---
  p-value: 0.0000
  → STATUS: STATIONARY (Safe for baseline regressions!)



In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_report(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"--- ADF Test: {name} ---")
    print(f"  p-value: {result[1]:.4f}")
    if result[1] < 0.05:
        print("  → STATIONARY (Good for linear modeling)\n")
    else:
        print("  → NON-STATIONARY (Needs differencing)\n")

df = pd.read_csv("data/processed/final_feature_matrix.csv", index_col=0, parse_dates=True)

# Test price levels vs log returns
adf_report(df["eua_price"], "EUA Price (Levels)")
adf_report(df["d_log_eua"], "Δ Log EUA (Returns)")

--- ADF Test: EUA Price (Levels) ---
  p-value: 0.0000
  → STATIONARY (Good for linear modeling)

--- ADF Test: Δ Log EUA (Returns) ---
  p-value: 0.0000
  → STATIONARY (Good for linear modeling)

